# Cassava Leaf Disease — Inference Notebook

**Model**: BYOL-pretrained ResNet34, full fine-tune  
**Before running**: add your checkpoint as a Kaggle dataset and update `CHECKPOINT_PATH` below.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "/kaggle/input/cassava-byol-checkpoint/finetune_full_finetune_best.pt"
DATA_DIR        = "/kaggle/input/cassava-leaf-disease-classification"
OUTPUT_CSV      = "submission.csv"

# ── Hyperparameters ───────────────────────────────────────────────────────────
BACKBONE    = "resnet34"
IMG_SIZE    = 224
BATCH_SIZE  = 64
NUM_WORKERS = 2
NUM_CLASSES = 5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# ── Model definition ──────────────────────────────────────────────────────────

BACKBONE_FEATURE_DIM = {
    "resnet34":      512,
    "resnet50":      2048,
    "efficientnet_b4": 1792,
    "densenet121":   1024,
}

def build_backbone(name: str) -> nn.Module:
    if name == "resnet34":
        m = tvm.resnet34(weights=None)
        m.fc = nn.Identity()
    elif name == "resnet50":
        m = tvm.resnet50(weights=None)
        m.fc = nn.Identity()
    elif name == "efficientnet_b4":
        m = tvm.efficientnet_b4(weights=None)
        m.classifier = nn.Identity()
    elif name == "densenet121":
        m = tvm.densenet121(weights=None)
        m.classifier = nn.Identity()
    else:
        raise ValueError(f"Unknown backbone: {name}")
    return m


class LeafClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

In [ ]:
# ── Load checkpoint ───────────────────────────────────────────────────────────

state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu")

# Remap head keys: our checkpoint uses head.fc.* but we simplified to head.*
remapped = {}
for k, v in state_dict.items():
    remapped[k.replace("head.fc.", "head.")] = v

feature_dim = BACKBONE_FEATURE_DIM[BACKBONE]
backbone = build_backbone(BACKBONE)
model = LeafClassifier(backbone, feature_dim, NUM_CLASSES)
model.load_state_dict(remapped)
model.to(device)
model.eval()
print("Checkpoint loaded.")

In [ ]:
# ── Dataset & transforms ──────────────────────────────────────────────────────

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class TestDataset(Dataset):
    def __init__(self, image_ids, images_dir, transform=None):
        self.image_ids = image_ids
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img = Image.open(self.images_dir / image_id).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, image_id


sample_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
image_ids = sample_df["image_id"].tolist()

test_dataset = TestDataset(image_ids, f"{DATA_DIR}/test_images", transform=val_transform)
test_loader  = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
print(f"Test images: {len(test_dataset)}")

In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────

all_preds = []
all_ids   = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        preds  = model(images).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_ids.extend(ids)

print(f"Predicted {len(all_preds)} images.")

In [ ]:
# ── Write submission ──────────────────────────────────────────────────────────

submission = pd.DataFrame({"image_id": all_ids, "label": all_preds})
submission.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV}")
submission.head()